# Step 5.3 — MTE routing (all resonators, agent)

**Shunt** (0, 1, 4, 5): agent picks pad connector route (straight / L / 45 / Z).

**Series** (2, 3, 6, 7): agent picks outward offset **filled ring** — margin × band sweep informs fallback.

Output: **GDS only**. Inspect `BAW_MTE` (layer 5/0) in Virtuoso / KLayout.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import pandas as pd


def resolve_python_code_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ARTIFACTS = ROOT / "artifacts" / "mte_experiment"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import os
from src.agentic.series_mte import default_env_file, load_agent_env

ENV_FILE = ROOT / ".env"
loaded_env = load_agent_env(env_file=ENV_FILE, force=True)
api_key_set = bool(os.environ.get("ANTHROPIC_API_KEY"))

DRAFT.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"GDS out={DRAFT / 'MTE_experiment'}")
print(f"ENV file={ENV_FILE} exists={ENV_FILE.is_file()} loaded={loaded_env}")
print(f"Canonical env path={default_env_file()}")
print(f"ANTHROPIC_API_KEY set={api_key_set}")

ROOT=C:\Users\santosr4\Documents\rTEG Automation\python_code
GDS out=C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment
ENV file=C:\Users\santosr4\Documents\rTEG Automation\python_code\.env exists=True loaded=C:\Users\santosr4\Documents\rTEG Automation\python_code\.env
Canonical env path=C:\Users\santosr4\Documents\rTEG Automation\python_code\.env
ANTHROPIC_API_KEY set=True


## Pipeline (steps 1–4 + 5.1/5.2)

In [2]:
from src.layermap import load_layermap
from src.prep_resonator_ppd import prep_resonator_ppd
from src.prep_rteg_frame import prep_rteg_in_frame
from src.rteg_classify import ClassifyNodesConfig, classify_nodes
from src.rteg_collect import RtegCollectConfig, collect_geometry_roles
from src.separate import identify

FILTER = INPUT / "KB331_N_01.gds"
FRAME = INPUT / "KB331_N_Frame.gds"
GSG = INPUT / "GSG_frame.gds"

layermap = load_layermap(INPUT / "layermap")
id_result = identify(FILTER)
res_df = pd.DataFrame(id_result.resonator_rows())
ppd_assemblies = prep_resonator_ppd(res_df, id_result.resonators, GSG)
rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)

ALL_INDICES = tuple(range(len(rteg_assemblies)))
classifications: dict[int, object] = {}
roles_by_index: dict[int, object] = {}

for idx in ALL_INDICES:
    res = id_result.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx], res, id_result, layermap, RtegCollectConfig()
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        res_type=res.res_type,
        config=ClassifyNodesConfig(),
    )
    roles_by_index[idx] = roles
    classifications[idx] = classification

SERIES_INDICES = tuple(
    idx for idx in ALL_INDICES if id_result.resonators[idx].res_type == "series"
)
SHUNT_INDICES = tuple(
    idx for idx in ALL_INDICES if id_result.resonators[idx].res_type == "shunt"
)
print(f"shunt {SHUNT_INDICES} | series {SERIES_INDICES}")

shunt (0, 1, 4, 5) | series (2, 3, 6, 7)


## Experiment setup (contexts for all resonators)

In [3]:
from src.agentic.series_mte import SeriesMteExperimentConfig, build_series_mte_context

exp_cfg = SeriesMteExperimentConfig(
    artifacts_dir=ARTIFACTS / "series",
    output_gds_dir=DRAFT / "MTE_experiment",
    series_indices=SERIES_INDICES,
    shunt_indices=SHUNT_INDICES,
)
all_contexts = {
    idx: build_series_mte_context(
        idx,
        rteg_assemblies[idx],
        id_result.resonators[idx],
        id_result,
        classifications[idx],
        layermap,
    )
    for idx in (*SERIES_INDICES, *SHUNT_INDICES)
}
print(f"agent targets: series {exp_cfg.series_indices} | shunt {exp_cfg.shunt_indices}")

agent targets: series (2, 3, 6, 7) | shunt (0, 1, 4, 5)


## Series margin × band sweep (informs series fallback)

In [4]:
series_contexts = {idx: all_contexts[idx] for idx in SERIES_INDICES}

In [5]:
from src.agentic.series_mte import run_width_sweep

sweep_df = run_width_sweep(series_contexts, exp_cfg)
cols = [
    "index",
    "margin_um",
    "band_thickness_um",
    "apply_drc_finalize",
    "build_ok",
    "is_drc_clean",
    "min_ground_spacing_um",
    "body_overlap_fraction",
    "strip_area_um2",
    "mte_style",
]
sweep_df[sweep_df["build_ok"] == True][cols].sort_values(  # noqa: E712
    ["index", "margin_um", "band_thickness_um"]
)

,index,margin_um,band_thickness_um,apply_drc_finalize,build_ok,is_drc_clean,min_ground_spacing_um,body_overlap_fraction,strip_area_um2,mte_style
1,2,1.0,1.0,True,True,True,72.9,0.0,18.4,filled_ring
3,2,1.0,1.5,True,True,True,72.9,0.0,29.9,filled_ring
5,2,1.0,2.0,True,True,True,72.9,0.0,41.5,filled_ring
7,2,1.0,3.0,True,True,True,72.9,0.0,64.6,filled_ring
8,2,1.0,4.0,False,True,False,0.0,0.0,84.4,filled_ring
...,...,...,...,...,...,...,...,...,...,...
235,7,5.0,2.0,True,True,True,16.6,0.0,246.1,filled_ring
236,7,5.0,3.0,False,True,False,0.0,0.0,565.8,filled_ring
237,7,5.0,3.0,True,True,True,17.6,0.0,375.0,filled_ring
238,7,5.0,4.0,False,True,False,0.0,0.0,840.0,filled_ring


In [6]:
from src.agentic.series_mte import run_agents_for_indices

agent_runs = run_agents_for_indices(all_contexts, exp_cfg, sweep_df=sweep_df)

rows = []
for idx, run in sorted(agent_runs.items()):
    row = {
        "index": idx,
        "res_type": run.res_type,
        "source": run.source,
        "route": run.signal.connector.shape_name,
        "reaches_pad": run.signal.reaches_pad,
        "drc_clean": not run.signal.drc_violations,
        "min_spacing_um": run.signal.min_ground_spacing_um,
    }
    if run.series_result is not None:
        row["margin_um"] = run.series_result.margin_um
        row["band_um"] = run.series_result.band_thickness_um
        row["finalize"] = run.series_result.used_drc_finalize
    rows.append(row)

pd.DataFrame(rows)

,index,res_type,source,route,reaches_pad,drc_clean,min_spacing_um,margin_um,band_um,finalize
0,0,shunt,agent,route_L,True,True,68.390467,NaN,NaN,NaN
1,1,shunt,agent,route_L,True,True,57.352468,NaN,NaN,NaN
2,2,series,agent,on_resonator,False,True,72.872369,3.0,2.0,True
3,3,series,agent,on_resonator,False,True,21.087257,2.0,3.0,True
4,4,shunt,agent,route_L,True,True,98.926176,NaN,NaN,NaN
5,5,shunt,agent,route_L,True,False,0.000000,NaN,NaN,NaN
6,6,series,agent,on_resonator,False,True,16.382914,3.0,2.0,False
7,7,series,agent,on_resonator,False,True,19.558632,2.0,2.0,True


## Agent summary

In [7]:
for idx, run in sorted(agent_runs.items()):
    sig = run.signal
    if run.res_type == "series" and run.series_result is not None:
        r = run.series_result
        print(
            f"idx {idx} series: margin={r.margin_um}um band={r.band_thickness_um}um "
            f"drc_clean={r.is_drc_clean} ({run.source})"
        )
    else:
        print(
            f"idx {idx} shunt: route={sig.connector.shape_name} "
            f"reaches_pad={sig.reaches_pad} drc_clean={not sig.drc_violations} "
            f"({run.source})"
        )

idx 0 shunt: route=route_L reaches_pad=True drc_clean=True (agent)
idx 1 shunt: route=route_L reaches_pad=True drc_clean=True (agent)
idx 2 series: margin=3.0um band=2.0um drc_clean=True (agent)
idx 3 series: margin=2.0um band=3.0um drc_clean=True (agent)
idx 4 shunt: route=route_L reaches_pad=True drc_clean=True (agent)
idx 5 shunt: route=route_L reaches_pad=True drc_clean=False (agent)
idx 6 series: margin=3.0um band=2.0um drc_clean=True (agent)
idx 7 series: margin=2.0um band=2.0um drc_clean=True (agent)


## GDS export (all resonators)

In [8]:
from src.agentic.series_mte import assemble_experiment_signals, export_all_mte_gds

all_signals = assemble_experiment_signals(agent_runs)

all_results = export_all_mte_gds(
    rteg_assemblies,
    all_signals,
    layermap=layermap,
    output_dir=exp_cfg.output_gds_dir,
    parent=exp_cfg.parent_name,
)
for r in all_results:
    print(r.path)

C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_00_shuntq3_CDNS_781040784740_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_01_shuntq3_CDNS_781040784740_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_02_seriesq3_CDNS_781040784741_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_03_seriesq3_CDNS_781040784741_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_04_shuntq3_CDNS_781040784742_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_05_shuntq3_CDNS_781040784745_mte.gds
C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_experiment\KB331_N_01_RTEG1_06_seriesq3_CDNS_781040784747_mte.gds
C:\Users\santosr4\Documents\rTE